# Motif + chromVAR deviations


Three new tools port the ArchR motif / chromVAR pipeline:

- `epi.tl.add_motif_matrix` — scans JASPAR position-weight matrices against each peak with MOODS (pychromvar backend) and stores a sparse peak × motif binary matrix.
- `epi.tl.add_background_peaks` — per peak, draws k bias-matched peer peaks (`log10 accessibility, GC content`) using `chromVAR::getBackgroundPeaks`-style density-weighted sampling.
- `epi.tl.compute_deviations` — chromVAR-style per-cell motif deviation Z-scores. Implemented as a single sparse `M @ X` matmul per background iteration, so memory stays bounded regardless of cell count.

Cross-validated against ArchR + chromVAR on the hematopoiesis tutorial dataset (10 k cells × 141 k peaks × 746 motifs):

| Check | Value |
| --- | --- |
| per-TF Pearson r (epione Z vs ArchR Z across cells) | mean **0.94**, median **0.95** |
| top-variance TFs (GATA1::TAL1, CEBPA, BATF::JUN…) | **0.97–0.99** |
| top-100 variable-TF overlap | **98 / 100** |


## Data Preparation

We use snapatac2's bundled **pbmc5k** scATAC-seq as a fast demo (~1-2 min end-to-end). On larger datasets you'd follow the same API; `add_motif_matrix` scales with peaks × motifs and `compute_deviations` with peaks × cells × n_iterations.


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, pathlib
os.environ['XDG_CACHE_HOME'] = '/scratch/users/steorra/cache'

import numpy as np
import pandas as pd
import anndata as ad
import snapatac2 as snap
import epione as epi
import matplotlib.pyplot as plt
from IPython.display import display

epi.pl.plot_set()

WORK = pathlib.Path('/scratch/users/steorra/data/pbmc5k_chromvar')
WORK.mkdir(parents=True, exist_ok=True)


Import fragments → QC → tile matrix → peak matrix.


In [ ]:
%%time
h5 = WORK / 'pbmc5k.h5ad'
if h5.exists():
    data = snap.read(str(h5))
else:
    frag = snap.datasets.pbmc5k(type='fragment')
    data = snap.pp.import_fragments(
        fragment_file=str(frag),
        chrom_sizes=snap.genome.hg38,
        file=str(h5),
        sorted_by_barcode=False,
    )
    snap.metrics.tsse(data, snap.genome.hg38)
    snap.pp.filter_cells(data, min_counts=1000, min_tsse=5,
                         max_counts=100000)
    # Use a tile matrix as the peak matrix for the demo. For real
    # analyses you'd call peaks with macs3 and feed those peaks in.
    snap.pp.add_tile_matrix(data, bin_size=5000)
    snap.pp.select_features(data, n_features=50000)
data


In [ ]:
X   = data.X[:].tocsr()
obs = data.obs[:].to_pandas(); obs.index = list(data.obs_names[:])
var = data.var[:].to_pandas(); var.index = list(data.var_names[:])
adata = ad.AnnData(X=X, obs=obs, var=var)
if 'selected' in adata.var.columns:
    adata = adata[:, adata.var['selected'].astype(bool).values].copy()
adata.shape


## 1 · Scan motifs

pychromvar / MOODS both-strand scan, `p_value=5e-5` (chromVAR default). JASPAR 2020 CORE vertebrates is ~750 motifs.


In [ ]:
%%time
epi.tl.add_motif_matrix(
    adata,
    genome_fasta=str(snap.genome.hg38.fasta),
    motif_db='JASPAR2020',
    motif_collection='CORE',
    motif_tax_group=['vertebrates'],
    pvalue=5e-5,
)
adata.varm['motif'].shape, adata.varm['motif'].nnz


## 2 · Background peaks


In [ ]:
%%time
epi.tl.add_background_peaks(
    adata,
    n_iterations=50,
    genome_fasta=str(snap.genome.hg38.fasta),
    seed=1,
)
adata.varm['bg_peaks'].shape


## 3 · Per-cell motif deviations


In [ ]:
%%time
epi.tl.compute_deviations(
    adata,
    motif_key='motif',
    bg_key='bg_peaks',
)
adata.obsm['motif_deviations'].shape


### Top variable TFs

Rank motifs by Z-score variance across cells.


In [ ]:
Z = adata.obsm['motif_deviations']
names = np.asarray(adata.uns['motif_deviations_names'])
rank = np.argsort(-np.nanvar(Z, axis=0))
pd.DataFrame({
    'motif': names[rank[:15]],
    'variance': np.nanvar(Z, axis=0)[rank[:15]],
})


## Notes

- `add_motif_matrix` also accepts a user-supplied `pwms={name: 4×L}` dict for custom collections (cis-BP, HOCOMOCO…).
- `compute_deviations` writes both the bias-corrected Z-score (`obsm['motif_deviations']`) and the raw deviation (`obsm['motif_deviations_raw']`).
- For paired ATAC+RNA or richer plotting, hook the motif Z-score into a standard UMAP/Leiden workflow: `sc.pp.neighbors(adata, use_rep='motif_deviations')` + `sc.tl.leiden`.
